# 데이터 편향성 해결 (교과서 214~216쪽 · 심화 활동)

**시작하기 전에**
1. 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU → 저장** (학습을 두 번 하므로 GPU가 꼭 필요해요.)
2. 아래 **준비 셀**부터 실행해요. 「(더한 부분)」·「(고친 부분)」은 교과서 코드와 다른 줄이에요.
3. GPU가 잡히지 않으면 준비 셀만 실행하고, 회색조 학습(3·4)은 화면의 「셀 실행 흉내」로 따라가요.

사진 출처: 교과서 제공 실습 데이터(수업용)

## 0. 준비 — 산불 이미지 분류 모델 다시 만들기 (204~210쪽 코드를 한 셀로)
코랩은 창을 닫으면 내려받은 데이터와 학습한 모델이 사라져요. 「산불 이미지 분류하기」 노트북에서 한 일(데이터 내려받기 → 컬러 CNN 모델 학습)을 한 번에 다시 해요. GPU에서 1분 안팎 걸려요. 기다리는 동안 화면의 「데이터 편향성이란?」을 읽어요.

In [ ]:
# 사진 출처: 교과서 제공 실습 데이터(수업용)
# 산불 사진 묶음(fire.zip)이 있는 선생님 구글 드라이브 공유 주소 — 그대로 실행하면 돼요
ZIP_URL = 'https://drive.google.com/drive/folders/11MH6Ju4Eedq0veEMAvxT7lqP4bXzTdaY?usp=sharing'
import re
FID = re.search(r'[-\w]{25,}', ZIP_URL).group()   # 주소에서 ID 부분만 꺼내요
if '/folders/' in ZIP_URL:   # 폴더 주소를 받았을 때
    !gdown -q --folder "$FID" -O /content/dl && cp /content/dl/fire.zip fire.zip
else:                        # 파일(fire.zip) 주소를 받았을 때
    !gdown -q "$FID" -O fire.zip
!unzip -q -o fire.zip
!ls /content/fire
# ↑ fire  non  test1.jpg  test2.jpg  test3.jpg 가 보이면 성공! 안 보이면 선생님께 알려 주세요.
import os, glob, cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

path = '/content/fire'
train_fire = glob.glob(path + '/fire' + '/*')        # 산불 이미지 데이터
train_non_fire = glob.glob(path + '/non' + '/*')     # 산불이 아닌 이미지 데이터
print(len(train_fire), len(train_non_fire))

train_datagen = ImageDataGenerator(rescale = 1/255)  # RGB값을 0과 1 사이로 정규화
train_generator = train_datagen.flow_from_directory(
                  path, target_size = (400,400),
                  batch_size = 9, class_mode = 'binary',
                  seed = 3
)
model = tf.keras.models.Sequential([
      tf.keras.layers.Conv2D(16, (3, 3), activation = 'relu', input_shape = (400, 400, 3)),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(128, activation = 'relu'),
      tf.keras.layers.Dense(1, activation = 'sigmoid')
])
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
history = model.fit(train_generator, steps_per_epoch = 36, epochs = 8, verbose = 1)

## 1. 데이터 편향성 확인하기 (214쪽)
② 편향된 데이터로 데이터 편향성 확인하기 — **빨간색 건물** 이미지(test3.jpg)로 테스트해 보자.

결과가 0에 가까우면 산불 이미지(fire = 0), 1에 가까우면 산불이 아닌 이미지(non = 1)예요.

In [ ]:
img3 = Image.open(path + '/test3.jpg')              # 교과서: files.upload() 후 '/content/test3.jpg'
img_resize3 = img3.resize((400, 400))
test_img3 = image.img_to_array(img_resize3) / 255    # (더한 부분) 학습 데이터와 같은 0~1 범위로
test_img3 = np.expand_dims(test_img3, axis = 0)
images3 = np.vstack([test_img3])
classes3 = model.predict(images3, batch_size = 10)
print(classes3[0])                                   # e-14처럼 나오면 소수점 아래 0이 13개 = 거의 0
if classes3[0][0] < 0.5:                             # (고친 부분) 교과서: if classes3[0] == 0:
    print('Fire Image')
else:
    print('Non Fire Image')
img_resize3

## 2. 데이터 편향성 해결하기 (215쪽)
(gray_model 셀에서 `input_shape` 관련 경고(UserWarning)가 한 줄 나올 수 있어요. 오류가 아니니 그대로 진행해요.)

205쪽 코드의 `read_img`를 회색조(GRAY)로 바꿔 사진이 어떻게 보이는지 확인해요. (`cmap = 'gray'`를 넣어야 회색으로 보여요. 넣지 않으면 교과서 그림처럼 초록·보라색으로 보여요.)

In [ ]:
def read_img(file_path):
  img_arr = cv2.imread(file_path)
  return cv2.cvtColor(img_arr, cv2.COLOR_BGR2GRAY)   # BGR 컬러 이미지를 회색조로

fire_img = read_img(train_fire[7])
plt.subplot(1, 2, 1); plt.title('Fire'); plt.xlabel(fire_img.shape)
plt.imshow(fire_img, cmap = 'gray', aspect = 'auto')            # (더한 부분) cmap = 'gray'
non_fire_img = read_img(train_non_fire[7])
plt.subplot(1, 2, 2); plt.title('Non Fire'); plt.xlabel(non_fire_img.shape)
plt.imshow(non_fire_img, cmap = 'gray', aspect = 'auto')
plt.show()

### ⚠️ 여기서 잠깐! read_img만 바꾸면 학습 데이터는 그대로 컬러예요
`read_img`는 사진을 **화면에 보여 줄 때만** 쓰여요. 모델이 학습하는 데이터는 `flow_from_directory`가 폴더에서 다시 읽어 오고, 기본값이 컬러(`color_mode = 'rgb'`)예요. 그래서 **회색조로 학습하려면** 아래처럼
1. `flow_from_directory`에 `color_mode = 'grayscale'`을 넣고,
2. 모델 입력 모양을 `(400, 400, 1)`로 바꾸고 (색 채널이 3개 → 1개),
3. 테스트 사진도 `convert('L')`로 회색조로 바꿔야 해요.

In [ ]:
# (고친 부분) 회색조로 읽어 오는 훈련 데이터 설정
gray_datagen = ImageDataGenerator(rescale = 1/255)
gray_generator = gray_datagen.flow_from_directory(
                 path, target_size = (400,400),
                 batch_size = 9, class_mode = 'binary',
                 color_mode = 'grayscale',               # (더한 부분) 회색조로 읽기
                 seed = 3
)

In [ ]:
# CNN 모델 생성 — model이 아니라 gray_model이라는 다른 이름을 써서 헷갈리지 않게 해요
gray_model = tf.keras.models.Sequential([
      tf.keras.layers.Conv2D(16, (3, 3), activation = 'relu', input_shape = (400, 400, 1)),   # (고친 부분) 채널 1개
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(128, activation = 'relu'),
      tf.keras.layers.Dense(1, activation = 'sigmoid')
])
gray_model.summary()
gray_model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

## 3. 다시 학습하고 평가하기 (216쪽)
색상 특징이 없으므로 학습 횟수(epochs)를 8에서 **15**로 늘려요. GPU에서 1~2분 걸려요.

In [ ]:
history = gray_model.fit(gray_generator, steps_per_epoch = 36, epochs = 15, verbose = 1)

회색조로 학습한 모델로 **빨간색 건물(test3)과 진짜 산불(test1)을 둘 다** 테스트해요. 테스트 사진도 **회색조**로 바꿔서 넣어요. 빨간 건물만 맞히고 진짜 산불을 놓친다면 편향이 풀린 것이 아니에요.

In [ ]:
img_gray3 = img3.convert('L').resize((400, 400))     # (더한 부분) 테스트 이미지도 회색조로
test_gray3 = image.img_to_array(img_gray3) / 255      # 모양 (400, 400, 1)
test_gray3 = np.expand_dims(test_gray3, axis = 0)
images4 = np.vstack([test_gray3])
classes4 = gray_model.predict(images4, batch_size = 10)
print(classes4[0])                                    # (고친 부분) 교과서: print(classes2[0])
if classes4[0][0] < 0.5:
    print('Fire Image')
else:
    print('Non_Fire Image')
img_gray3

In [ ]:
# (더한 부분) 진짜 산불 사진(test1.jpg)도 회색조 모델로 확인하기
img1 = Image.open(path + '/test1.jpg')
img_gray1 = img1.convert('L').resize((400, 400))
test_gray1 = np.expand_dims(image.img_to_array(img_gray1) / 255, axis = 0)
classes5 = gray_model.predict(test_gray1, batch_size = 10)
print(classes5[0])
if classes5[0][0] < 0.5:
    print('Fire Image')
else:
    print('Non_Fire Image')
img_gray1

결과가 교과서(`[1.] Non_Fire Image`)와 다를 수도 있어요. 학습은 할 때마다 달라져요. 빨간 건물은 「산불 아님」으로 맞혔는데 **진짜 산불(test1)을 「산불 아님」으로 놓쳤다면**, 색 정보를 버린 대가예요(회색조 모델은 학습 정확도가 1.000에 가까워 학습 사진을 외워 버리기 쉬워요). 근본적인 해결은 **붉지만 산불이 아닌 사진(단풍·노을·빨간 건물)을 non 폴더에 더하는 것**이에요. 두 결과를 화면에 기록해요.

### (선택 · 시간이 남으면) 세 사진을 두 모델로 비교하기

In [ ]:
for name in ['test1.jpg', 'test2.jpg', 'test3.jpg']:
  img = Image.open(path + '/' + name)
  c = np.expand_dims(image.img_to_array(img.convert('RGB').resize((400, 400))) / 255, axis = 0)
  g = np.expand_dims(image.img_to_array(img.convert('L').resize((400, 400))) / 255, axis = 0)
  pc = model.predict(c, verbose = 0)[0][0]
  pg = gray_model.predict(g, verbose = 0)[0][0]
  print(name, '| 컬러 모델:', round(float(pc), 3), '| 회색조 모델:', round(float(pg), 3), '(0에 가까우면 산불)')